# Advanced RAG with LangChain
This notebook demonstrates a production-ready RAG pipeline with:
---

**Pre-processing:**
- **Pre-processing (Query Transformation):**- 
- Query transformation (rewriting, decomposition, step-back prompting)
- Adaptive RAG (routing queries to appropriate retrieval strategies)
 - **Query Rewriting**: Reformulates queries for better semantic matching
 - **Query Decomposition**: Breaks complex queries into sub-questions
 - **Step-back Prompting**: Generates broader context questions

  **Adaptive RAG:**
 - Routes queries to optimal retrieval strategies based on classification
 - Adjusts BM25/vector weights per query type
 
 **Post-processing (Reranking):**
 - **Cohere Rerank**: Neural reranker for semantic relevance
 - **ColBERT**: Token-level late interaction for precise matching
 - **Ensemble**: Weighted combination of both rerankers

**Memory Management:**
- Short-term memory with Redis TTL for recent interactions
- Long-term memory with Qdrant for cross-session knowledge retention
 
 The pipeline is production-ready with async batch processing and LangSmith tracing.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
#  Advanced RAG — Production Memory + Distributed Vector Search                                                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# Short-term memory  — Redis TTL-backed sliding window               ║
# Long-term memory   — Qdrant vector store (semantic episodic store)  ║
# DistributedVectorStore — unified Qdrant retriever (drop-in)         ║
# ProductionMemoryManager — orchestrates both tiers                   ║
# Enhanced __main__ — wires new layers, base pipeline unchanged   

In [ ]:
pip install python-dotenv  llama-index llama-index-retrievers-bm25 llama-index-llms-openai llama-index-embeddings-openai cohere qdrant-client redis

In [ ]:
from getpass import getpass
from dotenv import load_dotenv

import os

load_dotenv(override=True)  # Load environment variables from .env file

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")  # Set OPENAI_API_KEY from environment variable
if not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key:")
    
    
os.environ["QDRANT _API_KEY"] = os.getenv("COHERE_API_KEY", "")  # Set COHERE_API_KEY from environment variable
if not os.environ["COHERE_API_KEY"]:
    os.environ["COHERE_API_KEY"] = getpass("Enter OpenAI API key:")
    
os.environ["QDRANT API_KEY"] = os.getenv("QDRANT_API_KEY", "")  # Set OPENAI_API_KEY from environment variable
if not os.environ["QDRANT_API_KEY"]:
    os.environ["QDRANT_API_KEY"] = getpass("Enter OpenAI API key:")

In [ ]:
# -----------------------------
# DATA INGESTION
# ----------------------------

!mkdir -p 'data/paul_graham/'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt' -O 'data/paul_graham/paul_graham_essay.txt'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/refs/heads/main/docs/examples/data/paul_graham/paul_graham_essay_ambiguated.txt' -O 'data/paul_graham/paul_graham_essay-ambiguated.txt'

In [ ]:


import os, json, math, time
import tiktoken
from datetime import datetime, timedelta
from typing import Literal, Optional
from enum import Enum
from dataclasses import dataclass, field
from pydantic import BaseModel, Field
from collections import defaultdict

from llama_index.core import (
    Settings, VectorStoreIndex, SimpleDirectoryReader,
    StorageContext, load_index_from_storage,
)
from llama_index.core.schema import NodeWithScore
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
from langsmith import traceable
from colorama import Fore, Style

Settings.llm        = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
llm = Settings.llm

# =====================================================
# BUILD & INDEX SETUP
# =====================================================

# previous steps downloaded the data files, so we can proceed to build the index.
print(f"{Fore.GREEN}Data files: successfully downloaded{Fore.RESET}")

STORAGE_DIR = "./storage"
DATA_DIR    = "./data/paul_graham"

if not os.path.exists(STORAGE_DIR):
    documents = SimpleDirectoryReader(DATA_DIR).load_data()
    splitter  = SemanticSplitterNodeParser(embed_model=Settings.embed_model)
    nodes     = splitter.get_nodes_from_documents(documents)
    index     = VectorStoreIndex(nodes)
    index.storage_context.persist(STORAGE_DIR)
    print(f"{Fore.CYAN}Storage directory created and index persisted successfully.{Fore.RESET}")
else:
    storage_context = StorageContext.from_defaults(persist_dir=STORAGE_DIR)
    index = load_index_from_storage(storage_context)

all_nodes = list(index.docstore.docs.values())


In [ ]:
# ─────────────────────────────────────────────────────────
# EMBEDDING HELPER  (reuse existing OpenAI embed model)
# ─────────────────────────────────────────────────────────

class EmbeddingService:
    """
    Thin wrapper around LlamaIndex's embed_model so every layer shares
    the same model and avoids duplicate API calls.
    """
    def __init__(self, embed_model=None, dim: int = 1536):
        self._model = embed_model or Settings.embed_model
        self.dim    = dim   

    def embed(self, text: str) -> list[float]:
        return self._model.get_text_embedding(text)

    def embed_batch(self, texts: list[str]) -> list[list[float]]:
        return self._model.get_text_embedding_batch(texts)


_embedding_svc = EmbeddingService()

In [ ]:

# =============================================================================
# QUERY TRANSFORMATION — classify, rewrite, decompose
# =============================================================================


REWRITE_SYSTEM   = """You are a query optimization expert. Rewrite the user's query to be more
specific, clear, and optimized for semantic search retrieval.
Rules: expand abbreviations, add context terms, make implicit concepts explicit.
Return ONLY the rewritten query."""
HYDE_SYSTEM      = """You are an expert writer. Given a user question, generate a detailed answer
as if you already knew the answer. Return ONLY the generated answer."""
DECOMPOSE_SYSTEM = "Break the query into 2-4 simpler sub-questions. Return a JSON array of strings only."
STEPBACK_SYSTEM  = "Generate a more general step-back question. Return ONLY the step-back question."
ROUTER_SYSTEM    = """Classify the query:
- factual    Simple fact lookup
- analytical Complex multi-fact reasoning
- conceptual Abstract / how-why
- keyword    Exact terms / proper nouns
Return JSON: {\"query_type\": \"...\", \"reasoning\": \"...\"}"""

def rewrite_query(q):
    return llm.complete(f"{REWRITE_SYSTEM}\n\nOriginal: {q}").text.strip()

def hyde_generate(query: str) -> str:
    return llm.complete(f"{HYDE_SYSTEM}\n\nQuestion: {query}").text.strip()

def decompose_query(q):
    raw = llm.complete(f"{DECOMPOSE_SYSTEM}\n\nQuery: {q}").text.strip()
    return json.loads(raw.removeprefix("```json").removesuffix("```").strip())

def stepback_query(q):
    return llm.complete(f"{STEPBACK_SYSTEM}\n\nQuery: {q}").text.strip()

class QueryClassification(BaseModel):
    query_type: Literal["factual", "analytical", "conceptual", "keyword"]
    reasoning: str

def classify_query(q):
    raw  = llm.complete(f"{ROUTER_SYSTEM}\n\nQuery: {q}").text.strip()
    data = json.loads(raw.removeprefix("```json").removesuffix("```").strip())
    return QueryClassification(**data)


def classify_and_transform(query, verbose=False):
    """Classifies the query and applies appropriate transformations based on the classification."""
    classification = classify_query(query)
    if verbose:
        print(f"[Router]     {classification.query_type} — {classification.reasoning}")
        
        # Step 2 — optionally rewrite / decompose / step-back *(replace with HyDE for factual)
        if classification.query_type == "factual":
            retrieval_query = rewrite_query(query)
            if verbose: 
                print(f"[Rewrite]   {retrieval_query}")        

        elif classification.query_type in ("analytical", "conceptual"):
            sub_qs = decompose_query(query)
            retrieval_query = " ".join(sub_qs)  
            if verbose: print(f"[Decompose]  {sub_qs}")

        elif classification.query_type == "keyword":
            retrieval_query = query             
            if verbose: print(f"[Keyword]    keeping original")

        else:
            retrieval_query = query
        return retrieval_query, classification 

In [ ]:

# ─────────────────────────────────────────────────────────
#     HYBRID SEARCH PARAMETER TUNING
#     Static weights  → fixed per query-type (as before)
#     Dynamic weights → adjusted at query time using query
#                       features (length, has_numbers, etc.)
# ─────────────────────────────────────────────────────────

STATIC_WEIGHTS = {"factual": (0.8, 0.2), "analytical": (0.5, 0.5), "conceptual": (0.5, 0.5), "keyword": (0.3, 0.7)}

def _query_features(query):
    tokens = query.split()
    return {"length": len(tokens), "has_numbers": any(t.isdigit() for t in tokens),
            "has_quotes": '"' in query or "'" in query, "is_question": query.strip().endswith("?")}

def dynamic_weights(query, query_type):
    vec_w, bm25_w = STATIC_WEIGHTS.get(query_type, (0.6, 0.4))
    feats = _query_features(query)
    if feats["has_quotes"]:  bm25_w = min(bm25_w + 0.20, 0.95)
    if feats["length"] <= 3: bm25_w = min(bm25_w + 0.10, 0.95)
    if feats["has_numbers"]: vec_w  = min(vec_w  + 0.10, 0.95)
    total = vec_w + bm25_w
    return round(vec_w / total, 3), round(bm25_w / total, 3)

def tuned_hybrid_retrieve(query, query_type, top_k=5, use_dynamic=True, verbose=True):
    vec_w, bm25_w = dynamic_weights(query, query_type) if use_dynamic else STATIC_WEIGHTS.get(query_type, (0.6, 0.4))
    if verbose: print(f"[Weights]  vector={vec_w}  bm25={bm25_w}  ({'dynamic' if use_dynamic else 'static'})")
    vec_results  = index.as_retriever(similarity_top_k=top_k).retrieve(query)
    bm25_results = BM25Retriever.from_defaults(nodes=all_nodes, similarity_top_k=top_k).retrieve(query)
    scores, node_map = defaultdict(float), {}
    def _add(results, weight):
        max_s = max((n.score or 1e-9) for n in results) if results else 1.0
        for n in results:
            nid = n.node.node_id; scores[nid] += weight * (n.score or 0) / max_s; node_map[nid] = n
    _add(vec_results, vec_w); _add(bm25_results, bm25_w)
    return [node_map[nid] for nid, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]]




In [ ]:
# ─────────────────────────────────────────────────────────
#  RECIPROCAL RANK FUSION (RRF) + LEARNED FUSION
# ─────────────────────────────────────────────────────────

import cohere
co = cohere.Client(os.getenv("COHERE_API_KEY"))

def reciprocal_rank_fusion(result_lists, k=60):
    scores, node_map = defaultdict(float), {}
    for results in result_lists:
        for rank, nws in enumerate(results, 1):
            nid = nws.node.node_id
            scores[nid]  += 1.0 / (k + rank)
            node_map[nid] = nws
    fused = []
    for nid, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
        n = node_map[nid]; n.score = score; fused.append(n)
    return fused

def cohere_rerank(query: str, nodes, top_k: int = 5):
    """Rerank retrieved nodes with Cohere's reranking endpoint."""
    docs = [n.get_content() for n in nodes]

    response = co.rerank(
        query=query,
        documents=docs,
        top_n=top_k,
        model="rerank-english-v3.0"
    )

    reranked_nodes = []
    for r in response.results:
        node = nodes[r.index]
        node.score = r.relevance_score
        reranked_nodes.append(node)

    return reranked_nodes
    
class LearnedFusionWeights(BaseModel):
    vector_weight: float = 0.65
    bm25_weight:   float = 0.35
    def blend(self, vector_results, bm25_results):
        scores, node_map = defaultdict(float), {}
        def _add(results, weight):
            max_s = max((n.score or 0) for n in results) if results else 1.0
            for n in results:
                nid = n.node.node_id; scores[nid] += weight * (n.score or 0) / (max_s or 1); node_map[nid] = n
        _add(vector_results, self.vector_weight); _add(bm25_results, self.bm25_weight)
        fused = []
        for nid, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
            n = node_map[nid]; n.score = score; fused.append(n)
        return fused

def retrieve_with_rrf(query, top_k=5):
    vec  = index.as_retriever(similarity_top_k=top_k).retrieve(query)
    bm25 = BM25Retriever.from_defaults(nodes=all_nodes, similarity_top_k=top_k).retrieve(query)
    return reciprocal_rank_fusion([vec, bm25])[:top_k]

def retrieve_with_learned_fusion(query, top_k=5):
    vec  = index.as_retriever(similarity_top_k=top_k).retrieve(query)
    bm25 = BM25Retriever.from_defaults(nodes=all_nodes, similarity_top_k=top_k).retrieve(query)
    return LearnedFusionWeights().blend(vec, bm25)[:top_k]



In [ ]:

# ─────────────────────────────────────────────────────────
# HYBRID RAG + LONG-CONTEXT WINDOW ARCHITECTURE
# ─────────────────────────────────────────────────────────

llm_long = OpenAI(model="gpt-4o", temperature=0, max_tokens=4096)

def _group_nodes_by_doc(nodes):
    groups = defaultdict(list)
    for n in nodes:
        doc_id = n.node.ref_doc_id or n.node.node_id
        groups[doc_id].append(n)
    return groups

def hybrid_rag_long_context(query, top_k=5, long_context_threshold=3, verbose=True):
    vector_retriever = index.as_retriever(similarity_top_k=top_k)
    nodes            = vector_retriever.retrieve(query)
    groups           = _group_nodes_by_doc(nodes)
    deep_doc_ids     = [d for d, ns in groups.items() if len(ns) >= long_context_threshold]
    if deep_doc_ids and verbose:
        print(f"[LongCtx]  Deep-reading {len(deep_doc_ids)} document(s)")
    answers = []
    for doc_id in deep_doc_ids:
        doc_nodes = [n for n in all_nodes if (getattr(n, "ref_doc_id", None) or n.node_id) == doc_id]
        if not doc_nodes:
            doc_nodes = [n.node for n in groups[doc_id]]
        doc_nodes_sorted = sorted(doc_nodes, key=lambda n: getattr(n, "start_char_idx", 0) or 0)
        full_text = "\n".join(getattr(n, "text", getattr(n, "get_content", lambda: "")()) for n in doc_nodes_sorted)
        prompt = f"""You are performing deep document analysis.
Read the FULL document below and answer the question thoroughly.
Document:
{full_text[:110_000]}
Question: {query}
Answer:"""
        ans = llm_long.complete(prompt).text.strip()
        answers.append(f"[Deep analysis]\n{ans}")
    remaining = [n for n in nodes if (n.node.ref_doc_id or n.node.node_id) not in deep_doc_ids]
    if remaining:
        context = "\n\n".join(n.get_content() for n in remaining)
        prompt  = f"Answer using ONLY the context below.\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
        ans = llm.complete(prompt).text.strip()
        answers.append(f"[RAG answer]\n{ans}")
    return "\n\n---\n\n".join(answers) if answers else "No answer found."

In [ ]:

# =============================================================================
# CONTEXT OPTIMIZATION — PRUNING, SNAPSHOTTING & SUMMARIZATION
# =============================================================================

def _count_tokens(text: str) -> int:
    return len(tiktoken.encoding_for_model("gpt-4o-mini").encode(text))

@traceable()
def truncate_nodes(nodes: list, max_tokens: int = 1500) -> list:
    """Keep top nodes that fit within the token budget."""
    out, total = [], 0
    for n in nodes:
        t = _count_tokens(n.text)
        if total + t > max_tokens: break
        out.append(n); total += t
    return out

def build_context(nodes: list) -> str:
    """Join node texts; summarise with LLM if still too long."""
    text = "".join(n.text for n in nodes)
    if _count_tokens(text) > 1000:
        print("🧠 Summarizing context...")
        text = llm.complete("Summarize concisely, keeping all key facts:\n\n" + text).text.strip()
    else:
        print("Context fits within token limit, no summarisation needed.")
    return text



In [ ]:
# ─────────────────────────────────────────────────────────
#  CONVERSATION MEMORY MANAGER - episodic conversation memory with auto-summarization & snapshots
# ─────────────────────────────────────────────────────────

@dataclass
class MemoryTurn:
    role:      str
    content:   str
    timestamp: datetime = field(default_factory=datetime.utcnow)
    tokens:    int      = 0

class ConversationMemory:
    """
    Sliding-window memory with pruning, snapshots & summarisation.
    (Fixed snapshot routing bug)
    """
    def __init__(self, max_tokens=3_000, summarize_threshold=2_000, snapshot_dir="./snapshots"):
        self.turns               = []
        self.summary             = None
        self.max_tokens          = max_tokens
        self.summarize_threshold = summarize_threshold
        self.snapshot_dir        = snapshot_dir
        os.makedirs(snapshot_dir, exist_ok=True)

    def _count_tokens(self, text): return len(text.split())
    def _total_tokens(self):       return sum(t.tokens for t in self.turns)

    def add(self, role, content):
        """ Add a new turn to memory and trigger routing (summarization/pruning) if needed. """
        self.turns.append(MemoryTurn(role=role, content=content, tokens=self._count_tokens(content)))
        self.router()
        
        self.save_snapshot()

    def get_context(self):
        """ Get the current conversation context, including summary if available, formatted for prompting. """
        parts = []
        if self.summary: parts.append(f"[Summary of earlier conversation]\n{self.summary}")
        for t in self.turns: parts.append(f"{t.role.upper()}: {t.content}")
        return "\n\n".join(parts)

    def save_snapshot(self) -> str:
        os.makedirs(self.snapshot_dir, exist_ok=True)
        
        path = os.path.join(self.snapshot_dir, f"snap_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.json")
        abs_path = os.path.abspath(path) 
        
        with open(abs_path, "w") as f:
            json.dump({
                "summary": self.summary,
                "turns": [
                    {"role": t.role, "content": t.content,
                    "timestamp": t.timestamp.isoformat(), "tokens": t.tokens}
                    for t in self.turns
                ]
            }, f, indent=2)
        
        print(f"[Memory]  Snapshot saved → {abs_path}")
        return abs_path

    def load_snapshot(self, path):
        """ Load a snapshot from disk, replacing current memory state. """
        with open(path) as f: data = json.load(f)
        self.summary = data.get("summary")
        self.turns   = [MemoryTurn(role=t["role"], content=t["content"], timestamp=datetime.fromisoformat(t["timestamp"]), tokens=t["tokens"]) for t in data["turns"]]
        print(f"[Memory]   Snapshot loaded ← {path}")

    def router(self):
        """ Check if we need to summarise or prune the memory based on token counts. """
        if self._total_tokens() > self.summarize_threshold: self._summarize_old_turns()
        if self._total_tokens() > self.max_tokens:          self._prune()

    def _summarize_old_turns(self):
        """Summarize the oldest half of the conversation turns to reduce token count, keeping key facts and decisions."""
        if len(self.turns) < 4: return

       
        self.save_snapshot() 
        
        cutoff       = len(self.turns) // 2
        to_summarize = self.turns[:cutoff]; self.turns = self.turns[cutoff:]
        history_text = "\n".join(f"{t.role.upper()}: {t.content}" for t in to_summarize)
        if self.summary: history_text = f"[Previous summary]\n{self.summary}\n\n{history_text}"
        prompt = "Summarize the following conversation history concisely, preserving all key facts, decisions, and open questions.\n\n" + history_text
        self.summary = llm.complete(prompt).text.strip()
        print(f"{Fore.CYAN}[Memory]   Summarised {cutoff} turns → summary{Style.RESET_ALL}")

    def _prune(self):
        """Prune the oldest turns until we're under the max token limit. """
        before = len(self.turns)
        while self._total_tokens() > self.max_tokens and self.turns: self.turns.pop(0)
        print(f"{Fore.MAGENTA}[Memory]   Pruned {before - len(self.turns)} turn(s){Style.RESET_ALL}")

In [ ]:

# =============================================================================
# RETRIEVAL — hybrid BM25 + vector with multiple fusion strategies
# =============================================================================

_STATIC_WEIGHTS = {
    "factual":    (0.8, 0.2),
    "analytical": (0.5, 0.5),
    "conceptual": (0.5, 0.5),
    "keyword":    (0.3, 0.7),
}

def _vec_retriever(top_k=5):  return index.as_retriever(similarity_top_k=top_k)
def _bm25_retriever(top_k=5): return BM25Retriever.from_defaults(nodes=all_nodes, similarity_top_k=top_k)

def _rrf(lists: list[list[NodeWithScore]], k=60) -> list[NodeWithScore]:
    """Reciprocal Rank Fusion across any number of result lists."""
    scores, nmap = defaultdict(float), {}
    for lst in lists:
        for rank, n in enumerate(lst, 1):
            scores[n.node.node_id] += 1.0 / (k + rank)
            nmap[n.node.node_id]    = n
    for nid, s in scores.items(): nmap[nid].score = s
    return sorted(nmap.values(), key=lambda x: x.score or 0, reverse=True)

def _dynamic_weights(query: str, query_type: str) -> tuple[float, float]:
    """Nudge static weights based on query surface features."""
    vw, bw = _STATIC_WEIGHTS.get(query_type, (0.6, 0.4))
    tokens = query.split()
    if '"' in query or "'" in query: bw = min(bw + 0.20, 0.95)
    if len(tokens) <= 3:             bw = min(bw + 0.10, 0.95)
    if any(t.isdigit() for t in tokens): vw = min(vw + 0.10, 0.95)
    t = vw + bw
    return round(vw/t, 3), round(bw/t, 3)

def retrieve(query: str, query_type: str, method: str = "rrf", top_k: int = 5, verbose: bool = False) -> list[NodeWithScore]:
    """
    Unified hybrid retrieval entry point.
    method: "rrf" | "learned" | "dynamic"
    """
    vec  = _vec_retriever(top_k).retrieve(query)
    bm25 = _bm25_retriever(top_k).retrieve(query)
 
    if method == "rrf":
        return _rrf([vec, bm25])[:top_k]
 
    if method == "dynamic":
        vw, bw = _dynamic_weights(query, query_type)
        if verbose: print(f"[Weights]   vector={vw}  bm25={bw}  (dynamic)")
    else:  # "learned" — fixed offline-trained weights
        vw, bw = 0.65, 0.35
 
    scores, nmap = defaultdict(float), {}
    for lst, w in [(vec, vw), (bm25, bw)]:
        mx = max((n.score or 1e-9) for n in lst) if lst else 1.0
        for n in lst:
            scores[n.node.node_id] += w * (n.score or 0) / mx
            nmap[n.node.node_id]    = n
    return sorted(nmap.values(), key=lambda x: scores[x.node.node_id], reverse=True)[:top_k]
 
def cohere_rerank(query: str, nodes: list, top_k: int = 5) -> list:
    """Re-rank nodes with Cohere's cross-encoder."""
    res = co.rerank(query=query, documents=[n.get_content() for n in nodes],
                    top_n=top_k, model="rerank-english-v3.0")
    for r in res.results: nodes[r.index].score = r.relevance_score
    return [nodes[r.index] for r in res.results]

In [ ]:
# =====================================================
# ADAPTIVE RETRIEVAL (BM25 + Vector Search + Reranking (RRF) + Fusion) + Memory-augmented RAG
# =====================================================

RETRIEVER_WEIGHTS = {"factual": [0.8, 0.2], "keyword": [0.3, 0.7], "analytical": [0.5, 0.5], "conceptual": [0.5, 0.5]}

def get_adaptive_retriever(query_type, top_k=5):
    weights = RETRIEVER_WEIGHTS.get(query_type, [0.6, 0.4])
    return QueryFusionRetriever(
        retrievers=[index.as_retriever(similarity_top_k=top_k),
                    BM25Retriever.from_defaults(nodes=all_nodes, similarity_top_k=top_k)],
        retriever_weights=weights, num_queries=1, use_async=False,
    )

def adaptive_rag(query, verbose=True):
    """Full pipeline with adaptive retrieval based on query classification."""
    classification = classify_query(query)
    if verbose: print(f"[Router]     {classification.query_type} — {classification.reasoning}")
    if classification.query_type == "factual":
        retrieval_query = rewrite_query(query)
        if verbose: print(f"[Rewrite]   {retrieval_query}")
    elif classification.query_type in ("analytical", "conceptual"):
        sub_qs = decompose_query(query)
        retrieval_query = " ".join(sub_qs)
        if verbose: print(f"[Decompose]  {sub_qs}")
    elif classification.query_type == "keyword":
        retrieval_query = query
        if verbose: print("[Keyword]    keeping original")
    else:
        retrieval_query = query
    retriever = get_adaptive_retriever(classification.query_type)
    nodes     = retriever.retrieve(retrieval_query)
    context   = "\n\n".join(n.get_content() for n in nodes)
    answer_prompt = f"""Answer the question using ONLY the context below.

                    Context:
                    {context}

                    Question: {query}
                    Answer:"""
    return llm.complete(answer_prompt).text.strip()

memory = ConversationMemory()

def memory_augmented_adaptive_rag(query, metadata_filters=None, use_long_context=False, fusion_method="rrf", verbose=True):
    """
    Full pipeline with adaptive retrieval, fusion options, and explicit dual-checkpoint 
    memory snapshot saves (post-query and post-generation).
    """
    # ─── STEP 1: LOG USER TURN & SAVE SNAPSHOT ───────────────────────────
    memory.add(role="user", content=query)
    if verbose: print(f"[Checkpoint] Saving pre-generation snapshot...")
    memory.save_snapshot() 
    
    # ─── STEP 2: CLASSIFY & TRANSFORM RETRIEVAL TARGETS ─────────────────
    cls = classify_query(query)
    if verbose: print(f"[Router]    {cls.query_type} — {cls.reasoning}")
    
    if cls.query_type == "factual":
        retrieval_query = rewrite_query(query)
    elif cls.query_type in ("analytical", "conceptual"):
        retrieval_query = " ".join(decompose_query(query))
    else:
        retrieval_query = query
        
    if verbose: print(f"[Query]     {retrieval_query}")
    
    # ─── STEP 3: EXECUTE HYBRID ENGINE AND FUSION MATCHING ─────────────
    retrievers = {
        "rrf":     lambda q: retrieve_with_rrf(q),
        "learned": lambda q: retrieve_with_learned_fusion(q),
        "dynamic": lambda q: tuned_hybrid_retrieve(q, cls.query_type, verbose=verbose),
    }
    
    nodes = retrievers.get(fusion_method, retrievers["rrf"])(retrieval_query)
    if verbose: print(f"[Filter]    {len(nodes)} nodes retrieved after blending.")
    
    # ─── STEP 4: GENERATE PRODUCTION ANSWER ──────────────────────────────
    if use_long_context:
        answer = hybrid_rag_long_context(query, verbose=verbose)
    else:
        # Build prompt using the unified sliding window conversation layer
        context_str = "\n\n".join(n.get_content() for n in nodes)
        prompt = f"""Use the context and conversation history to answer the question.
Conversation history:
{memory.get_context()}

Context:
{context_str}

Question: {query}
Answer:"""
        answer = llm.complete(prompt).text.strip()
    
    # ─── STEP 5: LOG LLM ANSWER & RE-SAVE TRANSACTION SNAPSHOT ─────────
    memory.add(role="assistant", content=answer)
    if verbose: print(f"[Checkpoint] Saving post-generation snapshot...")
    memory.save_snapshot()
    
    return answer

In [ ]:

import uuid
import hashlib
import logging
import threading
from abc import ABC, abstractmethod
from typing import Any

import numpy as np

logger = logging.getLogger(__name__)

# ─────────────────────────────────────────────────────────
# DEPENDENCY GUARD  — graceful fallback when Redis / Qdrant
# are not installed or servers are unreachable
# ─────────────────────────────────────────────────────────

def _try_import(package: str, install_hint: str):
    try:
        import importlib
        return importlib.import_module(package)
    except ImportError:
        logger.warning(f"[deps] '{package}' not installed. Run: {install_hint}")
        return None

redis_lib    = _try_import("redis",    "pip install redis")
qdrant_lib   = _try_import("qdrant_client", "pip install qdrant-client")




**Configuration: short and long-term memory**

In [ ]:
REDIS_URL   = os.getenv("REDIS_URL",   "redis://localhost:6379/0")
QDRANT_URL  = os.getenv("QDRANT_ENDPOINT",  f"http://localhost:6333")
QDRANT_KEY  = os.getenv("QDRANT_API_KEY", None)
SESSION_ID  = os.getenv("RAG_SESSION_ID", f"session-{datetime.utcnow().strftime('%Y%m%d-%H%M')}")

print(f"\n{'='*60}")
print(f"  Production RAG Pipeline")
print(f"  Session : {SESSION_ID}")
print(f"  Redis   : {REDIS_URL}")
print(f"  Qdrant  : {QDRANT_URL}")
print(f"{'='*60}\n")

In [ ]:

# ─────────────────────────────────────────────────────────
# SHORT-TERM MEMORY  — Redis TTL store
# ─────────────────────────────────────────────────────────


class RedisShortTermMemory:
    """
    Redis-backed short-term memory.
    Mirrors ConversationMemory's public interface so it can be used
    as a drop-in inside ProductionMemoryManager.
    Falls back to in-process dict if Redis is unavailable.
    """

    DEFAULT_TTL_SECONDS = 86_400   # 24 hours
    KEY_PREFIX          = "rag:stm"

    def __init__(
        self,
        session_id:       str   = "default",
        redis_url:        str   = "redis://localhost:6379/0",
        ttl_seconds:      int   = DEFAULT_TTL_SECONDS,
        max_turns:        int   = 20,
        summarize_at:     int   = 15,
    ):
        self.session_id   = session_id
        self.ttl          = ttl_seconds
        self.max_turns    = max_turns
        self.summarize_at = summarize_at

        self._client = None
        self._fallback: dict = {"turns": [], "summary": None}  # in-process fallback

        if redis_lib:
            try:
                self._client = redis_lib.Redis.from_url(redis_url, decode_responses=True)
                self._client.ping()
                print(f"{Fore.GREEN}[STM]  Redis connected → {redis_url}{Style.RESET_ALL}")
            except Exception as e:
                logger.warning(f"[STM]  Redis unavailable ({e}), using in-process fallback.")
                self._client = None
        else:
            logger.warning("[STM]  redis library not installed — using in-process fallback.")

    # ── key helpers ────────────────────────────────────────
    def _k(self, suffix: str) -> str:
        return f"{self.KEY_PREFIX}:{self.session_id}:{suffix}"

    # ── Redis ↔ fallback I/O ────────────────────────────────
    def _get(self, suffix: str, default=None):
        if self._client:
            try:
                raw = self._client.get(self._k(suffix))
                return json.loads(raw) if raw else default
            except Exception as e:
                logger.warning(f"[STM] Redis GET error: {e}")
        return self._fallback.get(suffix, default)

    def _set(self, suffix: str, value) -> None:
        serialised = json.dumps(value, default=str)
        if self._client:
            try:
                self._client.set(self._k(suffix), serialised, ex=self.ttl)
                return
            except Exception as e:
                logger.warning(f"[STM] Redis SET error: {e}")
        self._fallback[suffix] = value

    # ── public API ─────────────────────────────────────────
    @property
    def turns(self) -> list[dict]:
        return self._get("turns", [])

    @property
    def summary(self) -> Optional[str]:
        return self._get("summary")

    def add(self, role: str, content: str) -> None:
        """Append a turn, then manage window size."""
        turns = self.turns
        turns.append({
            "role":      role,
            "content":   content,
            "timestamp": datetime.utcnow().isoformat(),
            "tokens":    len(content.split()),
        })
        self._set("turns", turns)
        self._refresh_meta()
        self._maybe_manage()

    def get_context(self, max_turns: int = 10) -> str:
        """Return the most recent turns as a formatted string."""
        parts  = []
        turns  = self.turns[-max_turns:]
        summary = self.summary
        if summary:
            parts.append(f"[Short-term summary]\n{summary}")
        for t in turns:
            parts.append(f"{t['role'].upper()}: {t['content']}")
        return "\n\n".join(parts)

    def flush(self) -> None:
        """Delete this session's keys (e.g. on logout)."""
        if self._client:
            for suffix in ("turns", "summary", "meta"):
                try:
                    self._client.delete(self._k(suffix))
                except Exception:
                    pass
        else:
            self._fallback = {"turns": [], "summary": None}
        print(f"[STM]  Session {self.session_id} flushed.")

    def to_snapshot_dict(self) -> dict:
        return {"turns": self.turns, "summary": self.summary, "session_id": self.session_id}

    # ── internal management ────────────────────────────────
    def _refresh_meta(self) -> None:
        meta = self._get("meta", {"created_at": datetime.utcnow().isoformat(), "query_cnt": 0})
        meta["query_cnt"] = meta.get("query_cnt", 0) + 1
        meta["last_seen"] = datetime.utcnow().isoformat()
        self._set("meta", meta)

    def _total_tokens(self) -> int:
        return sum(t.get("tokens", 0) for t in self.turns)

    def _maybe_manage(self) -> None:
        turns = self.turns
        if len(turns) >= self.summarize_at:
            self._summarize_and_compress(turns)
        elif len(turns) > self.max_turns:
            self._set("turns", turns[-self.max_turns:])

    def _summarize_and_compress(self, turns: list[dict]) -> None:
        """Summarise oldest 50% of turns, store summary, keep recent half."""
        cutoff   = len(turns) // 2
        old_text = "\n".join(f"{t['role'].upper()}: {t['content']}" for t in turns[:cutoff])
        prior    = self.summary or ""
        if prior:
            old_text = f"[Prior summary]\n{prior}\n\n{old_text}"
        prompt      = "Summarise this conversation concisely, keeping all key facts.\n\n" + old_text
        new_summary = llm.complete(prompt).text.strip()
        self._set("summary", new_summary)
        self._set("turns", turns[cutoff:])
        print(f"{Fore.CYAN}[STM]  Compressed {cutoff} turns → Redis summary{Style.RESET_ALL}")

In [ ]:

# ─────────────────────────────────────────────────────────
# LONG-TERM MEMORY  — Qdrant episodic store
# ─────────────────────────────────────────────────────────

class QdrantLongTermMemory:
    """
    Qdrant-backed long-term episodic memory.
    Semantically retrieves relevant past conversations given a query.
    Falls back to in-process list when Qdrant is unavailable.
    """

    COLLECTION   = "rag_long_term_memory"
    VECTOR_SIZE  = 1536   # text-embedding-3-small

    def __init__(
        self,
        qdrant_url:  str  = "http://localhost:6333",
        api_key:     str  = None,
        top_k:       int  = 5,
        score_threshold: float = 0.70,
    ):
        self.top_k           = top_k
        self.score_threshold = score_threshold
        self._client         = None
        self._fallback: list[dict] = []   # in-process store

        if qdrant_lib:
            try:
                from qdrant_client import QdrantClient, models as qmodels
                self._client  = QdrantClient(url=qdrant_url, api_key=api_key, timeout=5)
                self._qmodels = qmodels
                self._ensure_collection()
                print(f"{Fore.GREEN}[LTM]  Qdrant connected → {qdrant_url}{Style.RESET_ALL}")
            except Exception as e:
                logger.warning(f"[LTM]  Qdrant unavailable ({e}), using in-process fallback.")
                self._client = None
        else:
            logger.warning("[LTM]  qdrant-client not installed — using in-process fallback.")

    def _ensure_collection(self) -> None:
        """Create the Qdrant collection if it doesn't exist."""
        existing = {c.name for c in self._client.get_collections().collections}
        if self.COLLECTION not in existing:
            self._client.create_collection(
                collection_name=self.COLLECTION,
                vectors_config=self._qmodels.VectorParams(
                    size=self.VECTOR_SIZE,
                    distance=self._qmodels.Distance.COSINE,
                ),
            )
            # Index on session_id for fast filtered retrieval
            self._client.create_payload_index(
                collection_name=self.COLLECTION,
                field_name="session_id",
                field_schema=self._qmodels.PayloadSchemaType.KEYWORD,
            )
            print(f"[LTM]  Created Qdrant collection '{self.COLLECTION}'")

    # ── public API ─────────────────────────────────────────
    def store(self, session_id: str, role: str, content: str, query: str = "") -> None:
        """Embed and upsert a turn into long-term memory."""
        vector  = _embedding_svc.embed(content)
        payload = {
            "session_id": session_id,
            "role":       role,
            "content":    content,
            "timestamp":  datetime.utcnow().isoformat(),
            "query":      query,
        }
        if self._client:
            try:
                self._client.upsert(
                    collection_name=self.COLLECTION,
                    points=[self._qmodels.PointStruct(
                        id=str(uuid.uuid4()), vector=vector, payload=payload
                    )],
                )
                return
            except Exception as e:
                logger.warning(f"[LTM] Qdrant upsert error: {e}")
       
        self._fallback.append({"vector": vector, "payload": payload})

    def retrieve_relevant(
        self,
        query:      str,
        session_id: str   = None,
        top_k:      int   = None,
    ) -> list[dict]:
        """
        Semantic search over long-term memory.
        Optionally filters to a single session.
        Returns list of payload dicts, highest-similarity first.
        """
        top_k  = top_k or self.top_k
        vector = _embedding_svc.embed(query)

        if self._client:
            try:
                query_filter = None
                if session_id:
                    query_filter = self._qmodels.Filter(
                        must=[self._qmodels.FieldCondition(
                            key="session_id",
                            match=self._qmodels.MatchValue(value=session_id),
                        )]
                    )
                hits = self._client.search(
                    collection_name=self.COLLECTION,
                    query_vector=vector,
                    query_filter=query_filter,
                    limit=top_k,
                    score_threshold=self.score_threshold,
                )
                return [h.payload for h in hits]
            except Exception as e:
                logger.warning(f"[LTM] Qdrant search error: {e}")

        # fallback: cosine similarity over in-process list
        if not self._fallback:
            return []
        q_vec = np.array(vector)
        scored = []
        for entry in self._fallback:
            v    = np.array(entry["vector"])
            cos  = float(np.dot(q_vec, v) / (np.linalg.norm(q_vec) * np.linalg.norm(v) + 1e-10))
            if cos >= self.score_threshold:
                scored.append((cos, entry["payload"]))
        scored.sort(reverse=True, key=lambda x: x[0])
        return [p for _, p in scored[:top_k]]

    def get_episodic_context(self, query: str, session_id: str = None) -> str:
        """Format retrieved long-term memories as a context string."""
        memories = self.retrieve_relevant(query, session_id=session_id)
        if not memories:
            return ""
        parts = ["[Relevant long-term memories]"]
        for m in memories:
            ts   = m.get("timestamp", "")[:10]
            role = m.get("role", "?").upper()
            parts.append(f"  [{ts}] {role}: {m['content'][:300]}")
        return "\n".join(parts)

    def store_summary(self, session_id: str, summary: str, query: str = "") -> None:
        """Store a compressed session summary as a long-term memory point."""
        self.store(session_id, role="summary", content=summary, query=query)

In [ ]:

# ─────────────────────────────────────────────────────────
# DISTRIBUTED VECTOR STORE  — Qdrant for RAG retrieval
# ─────────────────────────────────────────────────────────

class QdrantVectorStore:
    """Production-grade distributed vector store backed by Qdrant."""

    COLLECTION  = "rag_documents"
    VECTOR_SIZE = 1536
    BATCH_SIZE  = 64

    def __init__(
        self,
        qdrant_url: str   = "http://localhost:6333",
        api_key:    str   = None,
        top_k:      int   = 5,
    ):
        self.top_k   = top_k
        self._client = None
        self._fallback_nodes: list[NodeWithScore] = []   # in-process fallback

        if qdrant_lib:
            try:
                from qdrant_client import QdrantClient, models as qmodels
                self._client  = QdrantClient(url=qdrant_url, api_key=api_key, timeout=10)
                self._qmodels = qmodels
                self._ensure_collection()
                print(f"{Fore.GREEN}[VecStore] Qdrant distributed store ready → {qdrant_url}{Style.RESET_ALL}")
            except Exception as e:
                logger.warning(f"[VecStore] Qdrant unavailable ({e}) — using in-process fallback.")
                self._client = None
        else:
            logger.warning("[VecStore] qdrant-client not installed — using in-process fallback.")

    def _ensure_collection(self) -> None:
        existing = {c.name for c in self._client.get_collections().collections}
        if self.COLLECTION not in existing:
            self._client.create_collection(
                collection_name=self.COLLECTION,
                vectors_config=self._qmodels.VectorParams(
                    size=self.VECTOR_SIZE,
                    distance=self._qmodels.Distance.COSINE,
                ),
            )
            # Payload indexes for fast filtered retrieval
            for field, schema in [
                ("doc_id", self._qmodels.PayloadSchemaType.KEYWORD),
                ("source", self._qmodels.PayloadSchemaType.KEYWORD),
            ]:
                self._client.create_payload_index(
                    collection_name=self.COLLECTION,
                    field_name=field,
                    field_schema=schema,
                )
            print(f"[VecStore] Created Qdrant collection '{self.COLLECTION}'")

    # ── indexing ───────────────────────────────────────────
    def index_nodes(self, nodes) -> None:
        """
        Embed and upsert LlamaIndex nodes into Qdrant.
        Accepts raw BaseNode objects or NodeWithScore wrappers.
        """
        raw_nodes = [n.node if isinstance(n, NodeWithScore) else n for n in nodes]
        texts     = [n.get_content() for n in raw_nodes]
        vectors   = _embedding_svc.embed_batch(texts)

        points = []
        for node, vec in zip(raw_nodes, vectors):
            points.append({
                "id":      str(uuid.uuid4()),
                "vector":  vec,
                "payload": {
                    "node_id":  node.node_id,
                    "text":     node.get_content()[:4000],
                    "doc_id":   node.ref_doc_id or node.node_id,
                    "source":   node.metadata.get("file_name", ""),
                    "metadata": json.dumps(node.metadata, default=str),
                },
            })

        if self._client:
            for i in range(0, len(points), self.BATCH_SIZE):
                batch = points[i : i + self.BATCH_SIZE]
                try:
                    self._client.upsert(
                        collection_name=self.COLLECTION,
                        points=[
                            self._qmodels.PointStruct(**p) for p in batch
                        ],
                    )
                except Exception as e:
                    logger.error(f"[VecStore] Qdrant upsert batch error: {e}")
            print(f"[VecStore] Indexed {len(points)} nodes into Qdrant.")
        else:
            # fallback: keep NodeWithScore list in memory
            from llama_index.core.schema import NodeWithScore as NWS, TextNode
            for node in raw_nodes:
                self._fallback_nodes.append(NodeWithScore(node=node, score=1.0))
            print(f"[VecStore] (fallback) Indexed {len(raw_nodes)} nodes in-process.")

    # ── retrieval ──────────────────────────────────────────
    def retrieve(
        self,
        query:           str,
        top_k:           int            = None,
        metadata_filter: Optional[dict] = None,
    ) -> list[NodeWithScore]:
        """
        Semantic retrieval from Qdrant.
        Returns a list of NodeWithScore compatible with all base-code functions.
        """
        top_k  = top_k or self.top_k
        vector = _embedding_svc.embed(query)

        if self._client:
            try:
                query_filter = self._build_filter(metadata_filter)
                hits = self._client.search(
                    collection_name=self.COLLECTION,
                    query_vector=vector,
                    query_filter=query_filter,
                    limit=top_k,
                )
                return [self._hit_to_node(h) for h in hits]
            except Exception as e:
                logger.warning(f"[VecStore] Qdrant search error: {e}")

        # fallback: cosine over in-process nodes
        return self._fallback_retrieve(vector, top_k)

    def _build_filter(self, metadata_filter: Optional[dict]):
        if not metadata_filter or not self._client:
            return None
        conditions = [
            self._qmodels.FieldCondition(
                key=k,
                match=self._qmodels.MatchValue(value=str(v)),
            )
            for k, v in metadata_filter.items()
        ]
        return self._qmodels.Filter(must=conditions)

    def _hit_to_node(self, hit) -> NodeWithScore:
        """Convert a Qdrant ScoredPoint to a LlamaIndex NodeWithScore."""
        from llama_index.core.schema import TextNode
        payload = hit.payload or {}
        node    = TextNode(
            text=payload.get("text", ""),
            id_=payload.get("node_id", str(hit.id)),
            metadata=json.loads(payload.get("metadata", "{}")),
        )
        return NodeWithScore(node=node, score=hit.score)

    def _fallback_retrieve(self, vector: list[float], top_k: int) -> list[NodeWithScore]:
        if not self._fallback_nodes:
            return []
        q_vec = np.array(vector)
        scored = []
        for nws in self._fallback_nodes:
            text  = nws.node.get_content()
            v     = np.array(_embedding_svc.embed(text))
            cos   = float(np.dot(q_vec, v) / (np.linalg.norm(q_vec) * np.linalg.norm(v) + 1e-10))
            scored.append((cos, nws))
        scored.sort(reverse=True, key=lambda x: x[0])
        results = []
        for cos, nws in scored[:top_k]:
            nws.score = cos
            results.append(nws)
        return results

In [ ]:

# ─────────────────────────────────────────────────────────
# PRODUCTION MEMORY MANAGER
# Orchestrates short-term (Redis) + long-term (Qdrant)
# ─────────────────────────────────────────────────────────

class ProductionMemoryManager:
    """
    Unified memory facade.
    - Short-Term Memory-(Redis TTL store)
    - Long-Term Memory-(Qdrant episodic) 
    """

    # How many assistant turns to promote to long-term memory
    LTM_PROMOTE_EVERY_N = 3

    def __init__(
        self,
        session_id:   str  = "default",
        redis_url:    str  = "redis://localhost:6379/0",
        qdrant_url:   str  = os.getenv("QDRANT_ENDPOINT"),
        qdrant_key:   str  = os.getenv("QDRANT_API_KEY"),
        stm_max_turns: int = 20,
        ltm_top_k:    int  = 3,
    ):
        self.session_id     = session_id
        self._turn_counter  = 0

        self.stm = RedisShortTermMemory(
            session_id=session_id,
            redis_url=redis_url,
            max_turns=stm_max_turns,
        )
        self.ltm = QdrantLongTermMemory(
            qdrant_url=qdrant_url,
            api_key=qdrant_key,
            top_k=ltm_top_k,
        )

    # ── public API (mirrors ConversationMemory) ─────────────
    def add(self, role: str, content: str) -> None:
        """Add a turn to STM; periodically promote to LTM."""
        self.stm.add(role, content)
        self._turn_counter += 1

        # Promote assistant answers to long-term memory
        if role == "assistant" and self._turn_counter % self.LTM_PROMOTE_EVERY_N == 0:
            self.ltm.store(
                session_id=self.session_id,
                role=role,
                content=content,
                query=self._last_user_query(),
            )
            print(f"{Fore.YELLOW}[PMM]  Turn promoted to long-term memory (Qdrant){Style.RESET_ALL}")

        # Also promote STM summary to LTM when it is regenerated
        summary = self.stm.summary
        if summary:
            self.ltm.store_summary(self.session_id, summary)

    def get_context(self) -> str:
        """Short-term context (compatible with base full_pipeline)."""
        return self.stm.get_context()

    def get_combined_context(self, query: str) -> str:
        """
        Enriched context = short-term window  +  semantically relevant
        long-term memories for the current query.
        """
        stm_ctx = self.stm.get_context()
        ltm_ctx = self.ltm.get_episodic_context(query, session_id=self.session_id)

        parts = []
        if ltm_ctx:
            parts.append(ltm_ctx)
        if stm_ctx:
            parts.append(stm_ctx)
        return "\n\n".join(parts) if parts else ""

    def snapshot(self) -> str:
        """
        Delegate to the base ConversationMemory snapshot format
        (for compatibility with load_snapshot).
        Returns a JSON file path.
        """
        snap_dir = "./snapshots"
        os.makedirs(snap_dir, exist_ok=True)
        path = os.path.join(snap_dir, f"pmm_{self.session_id}_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.json")
        data = {
            "session_id": self.session_id,
            "summary":    self.stm.summary,
            "turns": self.stm.turns,
        }
        with open(path, "w") as f:
            json.dump(data, f, indent=2, default=str)
        print(f"[PMM]  Snapshot saved → {path}")
        return path

    def load_snapshot(self, path: str) -> None:
        """Load a JSON snapshot (compatible with ConversationMemory.load_snapshot)."""
        with open(path) as f:
            data = json.load(f)
        turns   = data.get("turns", [])
        summary = data.get("summary")
        self.stm._set("turns",   turns)
        self.stm._set("summary", summary)
        print(f"[PMM]  Snapshot loaded ← {path}")

    @property
    def summary(self) -> Optional[str]:
        return self.stm.summary

    # ── internal ───────────────────────────────────────────
    def _last_user_query(self) -> str:
        turns = self.stm.turns
        for t in reversed(turns):
            if t.get("role") == "user":
                return t.get("content", "")
        return ""

In [ ]:
# ─────────────────────────────────────────────────────────
#  DISTRIBUTED VECTOR STORE  — index existing nodes
# ─────────────────────────────────────────────────────────

def build_distributed_index(nodes, qdrant_url="http://localhost:6333") -> QdrantVectorStore:
    """
    One-time function to populate the Qdrant collection from the
    existing LlamaIndex node store.  Safe to call repeatedly
    (upsert semantics).
    """
    store = QdrantVectorStore(qdrant_url=qdrant_url)
    print(f"[VecStore] Indexing {len(nodes)} nodes into distributed store...")
    store.index_nodes(nodes)
    return store

In [ ]:
# =============================================================================
# GENERATION — LLM answer with context
# =============================================================================

@traceable()
def generate_answer(query: str, context: str) -> str:
    prompt = f"Answer using ONLY the context below.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    return llm.complete(prompt).text.strip()

In [ ]:
# =============================================================================
# ADAPTIVE RAG — stateful main pipeline (ProductionMemoryManager)
# =============================================================================

# Module-level singleton — ProductionMemoryManager orchestrates
_rag_memory = ProductionMemoryManager(                                 
    session_id=SESSION_ID,
    redis_url=REDIS_URL,
    qdrant_url=QDRANT_URL,
    qdrant_key=QDRANT_KEY,
)

def run_adaptive_rag(
    query:            str,
    fusion_method:    str  = "rrf",
    use_long_context: bool = False,
    metadata_filters: Optional[dict] = None,
    verbose:          bool = True,
    memory:           ProductionMemoryManager | ConversationMemory | None = None,  
) -> str:
    """
    End-to-end adaptive RAG pipeline — stateful with ProductionMemoryManager.

    Memory lifecycle per call
    ─────────────────────────
    1. Pull combined STM + LTM context for this query      (read)
    2. Append enriched context AFTER document context      (use)
    3. Save user query + assistant answer to STM           (write)
    4. Every 3rd assistant turn → promoted to Qdrant LTM   (promote)
    """

    # ── resolve which memory object to use ──────────────────────────────────
    mem = memory if memory is not None else _rag_memory

    # 1 — classify & transform
    print(f"{Fore.CYAN}Stage 1 (Query Transformation){Style.RESET_ALL}: \n")
    retrieval_query, cls = classify_and_transform(query, verbose=verbose)

    # 2 — hybrid retrieve + fusion
    nodes = retrieve(retrieval_query, cls.query_type, method=fusion_method, verbose=verbose)
    print(f"{Fore.CYAN}Stage 2 (Retrieval + Fusion){Style.RESET_ALL}: {len(nodes)} nodes\n")

    # 3 — Cohere rerank (analytical / conceptual only)
    if cls.query_type in ("analytical", "conceptual"):
        nodes = cohere_rerank(retrieval_query, nodes)
        print(f"[Rerank]   Cohere reranked → {len(nodes)} nodes")

    # 4 — metadata filter + token truncation
    if metadata_filters:
        nodes = [n for n in nodes if all(str(n.node.metadata.get(k)) == str(v)
                                         for k, v in metadata_filters.items())]
        if verbose: print(f"[Filter]   {len(nodes)} nodes after metadata filter")

    print(f"{Fore.CYAN}Stage 3 (Context Optimization): {Style.RESET_ALL}\n")
    doc_context = build_context(truncate_nodes(nodes, max_tokens=1200))


    if hasattr(mem, "get_combined_context"):
        memory_context = mem.get_combined_context(query)               
    else:
        memory_context = mem.get_context()                             

    if verbose and memory_context:
        turn_count = len(mem.stm.turns) if hasattr(mem, "stm") else len(mem.turns)
        token_count = mem.stm._total_tokens() if hasattr(mem, "stm") else mem._total()
        print(f"[Memory]   Injecting context — {turn_count} STM turn(s)"
              f" | ~{token_count} tokens")

    # 5 — generate
    if use_long_context:
        answer = hybrid_rag_long_context(query, verbose=verbose)
    else:
        if memory_context:
            prompt = (
                f"Answer using ONLY the context below.\n\n"
                f"Context:\n{doc_context}\n\n"
                f"Conversation history:\n{memory_context}\n\n"
                f"Question: {query}\nAnswer:"
            )
        else:
            # First turn — no history yet
            prompt = (
                f"Answer using ONLY the context below.\n\n"
                f"Context:\n{doc_context}\n\n"
                f"Question: {query}\nAnswer:"
            )
        answer = llm.complete(prompt).text.strip()

    mem.add("user",      query)
    mem.add("assistant", answer)                                        

    if verbose:
        if hasattr(mem, "stm"):
            print(f"[Memory]   STM={len(mem.stm.turns)} turns"
                  f" | session={mem.session_id}"
                  f" | LTM counter={mem._turn_counter}")
        else:
            print(f"[Memory]   {len(mem.turns)} turns | ~{mem._total()} tokens")

    print(f"{Fore.CYAN}\nAnswer{Style.RESET_ALL}:\n{answer}")
    return answer

In [ ]:

# =============================================================================
# ENTRY POINT: Run pipeline
# =============================================================================

def main():
    queries = [
        ("Summarize and analyze Paul Graham's essay 'Do What You Love'",                     "dynamic", True),
        ("Paul Graham advice on startup funding and investors",                               "rrf",     False),
        ("What are the key characteristics of successful founders according to Paul Graham?", "learned", False),
    ]
    
    
    # Snapshot the production memory manager (writes JSON to ./snapshots/)
    snap = _rag_memory.snapshot()                                       

    # Reload into a fresh PMM to verify snapshot round-trip
    fresh_memory = ProductionMemoryManager(                             
        session_id="fresh-verify",
        redis_url=REDIS_URL,
        qdrant_url=QDRANT_URL,
        qdrant_key=QDRANT_KEY,
    )
    fresh_memory.load_snapshot(snap)
    print(f"\nReloaded memory summary:\n{fresh_memory.stm.summary}")   

    for q, mode, long_ctx in queries:
        # Actively invoke the pipeline
        response = memory_augmented_adaptive_rag(
            query=q, 
            fusion_method="dynamic", 
            use_long_context=False, 
            verbose=True
        )
    
        print(f"\n{Fore.CYAN}🤖 LLM Response:{Fore.RESET}\n{response}\n")

        print(f"\n{'='*70}\nQuery : {q}\nMode  : {mode} | Long-context: {long_ctx}\n{'='*70}")
        answer = run_adaptive_rag(q, fusion_method=mode, use_long_context=long_ctx, verbose=True)
        print(f"\n{Fore.GREEN}Answer:\n{answer}{Style.RESET_ALL}\n")


**Run Pipeline**

In [ ]:
if __name__ == "__main__":
    main()